# Vector stores and semantic search



In [1]:
pip install sentence-transformers numpy pandas datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [3]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata

class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document

class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray = None

    def add_documents(self, documents: list[Document]):
        # Añadir documentos a la lista en memoria
        self.documents.extend(documents)
        
        # Extraer los textos y generar embeddings en batch
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(texts)
        
        # Apilar los nuevos embeddings en nuestra matriz NumPy
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack((self.embeddings, new_embeddings))

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        # 1. Vectorizar la consulta
        query_embedding = self.embedding_model.encode([query])[0]
        
        # 2. Calcular Similitud del Coseno vectorizada
        dot_products = np.dot(self.embeddings, query_embedding)
        norms_docs = norm(self.embeddings, axis=1)
        norm_query = norm(query_embedding)
        
        # Evitar división por cero
        norms_docs[norms_docs == 0] = 1e-10
        similarities = dot_products / (norms_docs * norm_query)
        
        # 3. Obtener los índices de los 'top_k' resultados más altos
        # argsort ordena de menor a mayor, por lo que invertimos [::-1]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        # 4. Construir y retornar los resultados
        return [SearchResult(float(similarities[idx]), self.documents[idx]) for idx in top_indices]

# ==========================================
# Ejecución y Pruebas - Parte I
# ==========================================

print("Cargando modelo...")
model = SentenceTransformer('all-MiniLM-L6-v2')
store = VectorStore(model)

print("Descargando Animal Fun Facts Dataset...")
url = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"
df_animals = pd.read_csv(url)

# Manejar valores nulos para evitar errores en metadatos
df_animals = df_animals.fillna("")

documents_to_add = []
for _, row in df_animals.iterrows():
    metadata = {
        "animal_name": str(row.get("animal_name", "")),
        "source": str(row.get("source", "")),
        "media_link": str(row.get("media_link", "")),
        "wikipedia_link": str(row.get("wikipedia_link", ""))
    }
    # Asumimos que la columna de texto se llama 'text' en el dataset original
    doc = Document(text=str(row.get("text", "")), metadata=metadata)
    documents_to_add.append(doc)

print(f"Indexando {len(documents_to_add)} documentos...")
store.add_documents(documents_to_add)

consultas_animales = [
    "What animal sleeps standing up?",
    "Birds that cannot fly but swim very well",
    "Which animal has the strongest bite force?",
    "Facts about marine mammals communicating",
    "Creatures that can change their skin color"
]

print("\n--- Resultados Parte I ---")
for q in consultas_animales:
    print(f"\nConsulta: '{q}'")
    resultados = store.search(q, top_k=2)
    for i, res in enumerate(resultados):
        animal = res.document.metadata.get('animal_name', 'Unknown')
        print(f"  [{i+1}] Score: {res.score:.4f} | Animal: {animal}")
        print(f"      Texto: {res.document.text[:100]}...")

Cargando modelo...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Descargando Animal Fun Facts Dataset...
Indexando 7734 documentos...

--- Resultados Parte I ---

Consulta: 'What animal sleeps standing up?'
  [1] Score: 0.6527 | Animal: elephant
      Texto: Elephants can sleep standing up and only need a couple of hours of sleep each day..
But not always, ...
  [2] Score: 0.6099 | Animal: camel
      Texto: Camels can sleep standing up..
While they usually sleep in a kneeling position with legs folded unde...

Consulta: 'Birds that cannot fly but swim very well'
  [1] Score: 0.7262 | Animal: bird
      Texto: Not all birds are able to fly!...
  [2] Score: 0.6803 | Animal: lamprey
      Texto: They technically can’t swim.
Despite living in water, their propulsion methods aren’t traditional....

Consulta: 'Which animal has the strongest bite force?'
  [1] Score: 0.6938 | Animal: thylacoleo
      Texto: Scientists have speculated that its bite force was equivalent to that of a 551 pound lion!...
  [2] Score: 0.6860 | Animal: tasmanian devil
      Text

## Part II: Filtering by metadata

In [4]:
from datasets import load_dataset

class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray = None

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(texts)
        
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack((self.embeddings, new_embeddings))

    def search(self, 
               query: str, 
               top_k: int = 5, 
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        
        if self.embeddings is None or len(self.documents) == 0:
            return []

        # PRE-FILTRADO: Obtener índices que cumplen el criterio
        valid_indices = []
        if metadata_filter:
            for idx, doc in enumerate(self.documents):
                match = True
                for key, val in metadata_filter.items():
                    if doc.metadata.get(key) != val:
                        match = False
                        break
                if match:
                    valid_indices.append(idx)
                    
            if not valid_indices:
                return [] # Ningún documento cumple el filtro
            
            target_embeddings = self.embeddings[valid_indices]
            target_docs = [self.documents[i] for i in valid_indices]
        else:
            target_embeddings = self.embeddings
            target_docs = self.documents

        # Operaciones de similitud solo sobre los embeddings filtrados
        query_embedding = self.embedding_model.encode([query])[0]
        
        dot_products = np.dot(target_embeddings, query_embedding)
        norms_docs = norm(target_embeddings, axis=1)
        norm_query = norm(query_embedding)
        norms_docs[norms_docs == 0] = 1e-10
        
        similarities = dot_products / (norms_docs * norm_query)
        
        # Obtener top_k de la sub-lista
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        return [SearchResult(float(similarities[idx]), target_docs[idx]) for idx in top_indices]

# ==========================================
# Ejecución y Pruebas - Parte II
# ==========================================

print("\nInicializando FilteredVectorStore...")
filtered_store = FilteredVectorStore(model)

print("Cargando AG News dataset (muestra de 1000 documentos para velocidad)...")
# Mapeo de etiquetas de AG News
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
dataset = load_dataset("ag_news", split="train[:1000]")

news_docs = []
for item in dataset:
    metadata = {
        "category": label_map[item["label"]],
    }
    # El texto de AG News
    doc = Document(text=item["text"], metadata=metadata)
    news_docs.append(doc)

print(f"Indexando {len(news_docs)} noticias con metadatos...")
filtered_store.add_documents(news_docs)

consultas_noticias = [
    {"q": "New discoveries in space exploration", "filter": {"category": "Sci/Tech"}},
    {"q": "Stock market drops significantly", "filter": {"category": "Business"}},
    {"q": "Championship final match results", "filter": {"category": "Sports"}},
    {"q": "Software update causes massive outage", "filter": {"category": "Sci/Tech"}},
    {"q": "Elections happening in Europe", "filter": {"category": "World"}}
]

print("\n--- Resultados Parte II (Con Filtros) ---")
for item in consultas_noticias:
    q = item["q"]
    f = item["filter"]
    print(f"\nConsulta: '{q}' | Filtro: {f}")
    resultados = filtered_store.search(q, top_k=2, metadata_filter=f)
    
    if not resultados:
        print("  Sin resultados que coincidan con el filtro.")
    
    for i, res in enumerate(resultados):
        cat = res.document.metadata.get('category', 'Unknown')
        print(f"  [{i+1}] Score: {res.score:.4f} | Categoría: {cat}")
        print(f"      Texto: {res.document.text[:100]}...")


Inicializando FilteredVectorStore...
Cargando AG News dataset (muestra de 1000 documentos para velocidad)...
Indexando 1000 noticias con metadatos...

--- Resultados Parte II (Con Filtros) ---

Consulta: 'New discoveries in space exploration' | Filtro: {'category': 'Sci/Tech'}
  [1] Score: 0.4967 | Categoría: Sci/Tech
      Texto: Redesigning Rockets: NASA Space Propulsion Finds a New Home (SPACE.com) SPACE.com - While the explor...
  [2] Score: 0.4650 | Categoría: Sci/Tech
      Texto: Space Science Pioneer Van Allen Questions Human Spaceflight (SPACE.com) SPACE.com - A leading space ...

Consulta: 'Stock market drops significantly' | Filtro: {'category': 'Business'}
  [1] Score: 0.5976 | Categoría: Business
      Texto: Stocks Fall as Oil Hits High (Reuters) Reuters - Exporters led a fall in Asian shares\on Monday as o...
  [2] Score: 0.5225 | Categoría: Business
      Texto: Stocks Fall as Oil Hits High  SINGAPORE (Reuters) - Exporters led a fall in Asian shares  on Monday ...

Con

## Reflexiones personas

### Parte una 

- Gran desempeno de las palabras claves: En la consulta "Stock market drops significantly", el sistema devolvio un articulo que dice "Stocks Fail as Oil Hits High". No hay coincidencia exacta de las palabras market, drops o significicantly. Pero el vector captura la intencion.

- Debilidad del modelo **all=MINILLM-L6** sus embeddings son relativamente pequenos. Por lo tanto pueden fallar en entender la direccion logica o la negacion. En la consulta relacionada a nadar y cannot, a pesar de que se pedia exactamente lo contrario. Siendo esto una limitacion arquitectonica.

### Parte dos

- Al limitar el dataset a 1,000 noticias. El modelo intento encontrar lo mas cercano matematicamente hablando. Relaciono voting y electino, pero al no encontrar noticias de elecciones en europa en la muestra. El motor matematico, siempre te dara un vecino mas cercano, incluso si ese vecino vive en otro continente.

### Importancia del pre-filtrado

- Al buscar los vectores mas cercanos, al identificar los mas cercanos. Elimina aquellos que no cumplen con los metadatos.

- Garantizamos mayor precision, tambien es util en entornos reales donde se utiliza en sistemas masivos con millones de vectores.







### 4. La elección del Dataset: 

* **Resolución de Ambigüedad Semántica:** En los sistemas reales, los vectores pueden ser víctimas de la polisemia (palabras con múltiples significados). Si un usuario busca el término *"Apple"*, el vector intentará buscar tanto la fruta como la compañía tecnológica. Al integrar un metadato duro como `{"category": "Sci/Tech"}` o `{"category": "Agriculture"}`, el sistema resuelve la ambigüedad instantáneamente.

* **Simulación de Agregadores Modernos:** Esta arquitectura (Búsqueda Vectorial + Filtro por Metadatos) es el esqueleto exacto de cómo funcionan plataformas como Google News o los feeds de recomendación. Permitimos al usuario hacer búsquedas difusas ("dame artículos sobre crisis económicas"), pero respetando restricciones de negocio estrictas ("pero solo muéstrame de la sección de Negocios, no de Política").

* **Optimización de Recursos (Pre-filtrado):** A nivel estructural, usar un dataset segmentado por categorías (Mundo, Deportes, Negocios, Ciencia) nos permitió demostrar que aplicar el filtro *antes* de la búsqueda reduce la matriz de embeddings. En lugar de hacer operaciones de álgebra lineal contra 120,000 vectores de noticias, las hacemos solo contra la fracción que pertenece a la categoría deseada, ahorrando memoria y ciclos de CPU.